In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/COURSES/DM_CSE426_Lab/Week 05 - Outlier Analysis II/employee_data.csv')
data

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ['Monthly_Sales', 'Customer_Satisfaction', 'Task_Completion_Rate', 'Absenteeism_Days', 'Overtime_Hours']

scaler = StandardScaler()

data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

data.head()

In [ ]:
from scipy.spatial.distance import mahalanobis

# Select the features for Mahalanobis distance calculation
features = ['Monthly_Sales', 'Customer_Satisfaction', 'Task_Completion_Rate', 'Absenteeism_Days', 'Overtime_Hours']
X = data[features]

# Calculate the mean vector
mean_vector = np.mean(X, axis=0)

# Calculate the covariance matrix
cov_matrix = np.cov(X, rowvar=False)

# Calculate the inverse of the covariance matrix
inv_cov_matrix = np.linalg.inv(cov_matrix)

In [ ]:
# Function to calculate Mahalanobis Distance

def calculate_mahalanobis(x):
    return mahalanobis(x, mean_vector, inv_cov_matrix)

In [ ]:
# Calculate Mahalanobis distances for all data points
mahalanobis_distances = X.apply(calculate_mahalanobis, axis=1)

data['Mahalanobis_Distance'] = mahalanobis_distances

outlier_threshold = 3
data['Mahalanobis_Distance_Outliers'] = (data['Mahalanobis_Distance'] > outlier_threshold)

data

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

# Initialize the Local Outlier Factor model
lof = LocalOutlierFactor(n_neighbors=20, contamination='auto')
lof.fit(X)

# Get the outlier scores
data['LOF_Score'] = lof.negative_outlier_factor_

# Identify outliers based on a threshold (e.g., scores below -1.5)
threshold_lof = -1.5
data['LOF_Outliers'] = (data['LOF_Score'] < threshold_lof)

data

In [ ]:
data['Combined_Outliers'] = data['Mahalanobis_Distance_Outliers'] & data['LOF_Outliers']

data[data['Combined_Outliers']]

**Isolation Forest**

In [ ]:
from sklearn.ensemble import IsolationForest

isolation_forest = IsolationForest(contamination='auto', random_state=42)
isolation_forest.fit(X)

# Predict outliers (-1 for outliers, 1 for inliers)
data['Isolation_Forest_Outliers'] = isolation_forest.predict(X)

# Convert predictions to boolean (1 for inliers, -1 for outliers)
data['Isolation_Forest_Outliers'] = data['Isolation_Forest_Outliers'] == -1

data[data['Isolation_Forest_Outliers']]

**One Class SVM**

In [ ]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(nu=0.1, kernel='rbf', gamma='scale')
ocsvm.fit(X)

# Predict outliers (-1 for outliers, 1 for inliers)
data['OCSVM_Outliers'] = ocsvm.predict(X)

# Convert predictions to boolean (True for outliers)
data['OCSVM_Outliers'] = data['OCSVM_Outliers'] == -1

data[data['OCSVM_Outliers']]

**DBSCAN Based Outliers**

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan.fit(X)

# Get the cluster labels
data['DBSCAN_Labels'] = dbscan.labels_

# Identify outliers (points assigned to the noise cluster, labeled -1)
data['DBSCAN_Outliers'] = data['DBSCAN_Labels'] == -1

data[data['DBSCAN_Outliers']]
